In [ ]:
%matplotlib ipympl

In [ ]:
from __future__ import annotations

import dataclasses
import functools
import typing as tp
import pandas as pd
import numpy as np
import jax
import jax.numpy as jnp
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import scipy.interpolate as sci_interp
import scipy.signal as sci_sig
import scipy.optimize as sci_opt
import scipy.fft as sci_fft

jax.config.update("jax_enable_x64", True)

In [ ]:
def load_clean_references(file_path: str) -> tuple[jax.Array, jax.Array]:
    data = np.array(pd.read_hdf(file_path))
    return data[:, 1:4], data[:, 4:]

file_path = "/Users/jozbee/work/eng/comp/data/clean_specific-forces-standard-road-v2.hdf"
acc_ref, omega_ref = load_clean_references(file_path)

acc_ref = jnp.clip(acc_ref, -1.0, 1.0)

In [ ]:
data_range = [0, 90 * 200]
data = acc_ref[data_range[0]: data_range[1], 0]

dt = 0.005
ts = np.arange(data_range[1] - data_range[0]) * dt
ref_data = sci_interp.make_smoothing_spline(ts, data, lam=1e0)(ts)

## future savgol

In [ ]:
# savgol_filt = sci_sig.savgol_filter(data, window_length=300, polyorder=2, axis=0)
savgol_filt = sci_sig.savgol_filter(data, window_length=200, polyorder=1, axis=0)
# savgol_filt = sci_sig.savgol_filter(data, window_length=150, polyorder=0, axis=0)

fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(data, label="data", alpha=0.4)
ax.plot(savgol_filt, label="savgol_filt")
ax.grid()

## sympy savgol

In [ ]:
def pred_savgol_coeffs(poly_deg: int, window_size: int, dt: float) -> jax.Array:
    """Result should be used with `np.convolve`"""
    if poly_deg == 0:
        # special case, where the solution is obviously the arithmetic mean
        return jnp.ones(window_size) / window_size

    n = poly_deg
    N = window_size

    t = sp.Symbol("t")
    dt_sp = sp.Symbol(r"\Delta t")
    a_s = []
    for k in range(n):
        a_s.append(sp.Symbol("a_{" + str(k) + "}"))
    us = []
    ts = []
    for k in range(N):
        us.append(sp.Symbol("u_{" + str(k) + "}"))
        ts.append(k * dt_sp)
    q = sum([a_s[k] * t**k for k in range(n)])

    alpha = sp.Symbol(r"\alpha")
    alpha *= 0  # maybe an interesting tuning parameter in the future
    J = sum([(q.subs(t, ts[k]) - us[k])**2 * sp.exp(-alpha * k * dt_sp) for k in range(N)])

    eqs = []
    for k in range(n):
        eqs.append(sp.Eq(J.diff(a_s[k]), 0))
    res = sp.solve(eqs, a_s, dict=True)
    a0 = res[0][a_s[0]].subs(dt_sp, dt)
    conv = jnp.array([float(a0.coeff(us[k])) for k in range(N)])
    return conv

In [ ]:
def pred_savgol_filter(data: jax.Array, window_length: int, polyorder: int, dt: float) -> jax.Array:
    coeffs = pred_savgol_coeffs(polyorder, window_length, dt)
    data = jnp.concatenate([jnp.zeros(window_length - 1), data])
    return jnp.convolve(data, coeffs, mode="valid")

# plot pred savgol

In [ ]:
# pred_savgol_filt = pred_savgol_filter(data, window_length=300, polyorder=0, dt=0.005)
pred_savgol_filt = pred_savgol_filter(data, window_length=200, polyorder=0, dt=0.005)

fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(data, label="data", alpha=0.4)
ax.plot(ref_data, label="ref_data", alpha=0.4)
ax.plot(pred_savgol_filt, label="pred_savgol_filt")
ax.legend()
ax.grid()